# Stage 9 — Hierarchical DDO2 Specification Freeze

This notebook is the **method and validation-protocol freeze**, not a final external validation and not a licence to tune on future targets. It verifies the immutable Stage 7/8/8B lineage, audits the 21 eligible directed edges from retinal fundus, dermoscopy, and chest radiography, freezes three separate diagnostic outputs (discrimination, calibration, and operating point), freezes a partially pooled modality-aware model family, defines dataset- and modality-grouped validation, and seals an explicit `ABSTAIN` policy.

The notebook is deliberately allowed to conclude that coefficient fitting is premature. It will never manufacture a fitted DDO2 when the pre-specified information gates are not met. It does not read images, download data, refit target models, tune thresholds, or inspect any future blind-test labels.


In [1]:
# @title 09-0. Mount Drive, verify immutable lineage, and seal the Stage 9 protocol
from google.colab import drive
drive.mount("/content/drive")

import hashlib
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

print("================ STAGE 9 HIERARCHICAL DDO2 SPECIFICATION PREFLIGHT ================")

PROJECT_ROOT = Path("/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability")
CODE_ROOT = PROJECT_ROOT / "05_Code" / "Cross_Modal"
RETINAL_ROOT = PROJECT_ROOT / "06_Data_Records" / "Retinal_DR" / "Prospective_Retinal_Blind_Test_v0.1"
CROSS_MODAL_ROOT = PROJECT_ROOT / "06_Data_Records" / "Cross_Modal"

STAGE7_ROOT = RETINAL_ROOT / "Stage7_PostUnseal_FourDomain_Transportability_Discovery_And_DDO2_Prototype_v0.1"
STAGE8_ROOT = CROSS_MODAL_ROOT / "Stage8_CrossModality_EdgeLibrary_Expansion_v0.1"
STAGE8B_ROOT = CROSS_MODAL_ROOT / "Stage8B_NLM_Chest_Radiography_Access_Completion_v0.1"
STAGE9_ROOT = CROSS_MODAL_ROOT / "Stage9_Hierarchical_DDO2_Specification_Freeze_v0.1"

PROTOCOL_ROOT = STAGE9_ROOT / "00_Protocol"
AUDIT_ROOT = STAGE9_ROOT / "01_Evidence_Audit"
SPEC_ROOT = STAGE9_ROOT / "02_Frozen_Specification"
VALIDATION_ROOT = STAGE9_ROOT / "03_Grouped_Validation_Protocol"
BLIND_ROOT = STAGE9_ROOT / "04_Future_Blind_Validation"
RESULT_ROOT = STAGE9_ROOT / "05_Results"
for directory in [CODE_ROOT, PROTOCOL_ROOT, AUDIT_ROOT, SPEC_ROOT, VALIDATION_ROOT, BLIND_ROOT, RESULT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = CODE_ROOT / "CrossModal_Stage9_Hierarchical_DDO2_Specification_Freeze_v0.1.ipynb"
STAGE7_FINAL_PATH = STAGE7_ROOT / "03_Results" / "Stage7_FourDomain_Discovery_Complete_v0.1.json"
STAGE7_CANDIDATE_PATH = STAGE7_ROOT / "02_DDO2_Prototype" / "Stage7_DDO2_Discovery_Candidates_v0.1.csv"
STAGE8_FINAL_PATH = STAGE8_ROOT / "06_Results" / "Stage8_CrossModality_Expansion_Complete_v0.1.json"
STAGE8B_FINAL_PATH = STAGE8B_ROOT / "06_Results" / "Stage8B_NLM_Access_Completion_Complete_v0.1.json"
STAGE8B_MANIFEST_PATH = STAGE8B_ROOT / "06_Results" / "Stage8B_Output_Integrity_Manifest_v0.1.csv"
EDGE_LIBRARY_PATH = STAGE8B_ROOT / "05_Unsealed_Chest_Discovery" / "Stage8B_ThreeModality_Eligible_Edge_Library_v0.1.csv"

PROTOCOL_SEAL_PATH = PROTOCOL_ROOT / "Stage9_Hierarchical_DDO2_Specification_Protocol_Seal_v0.1.json"
INPUT_COMMITMENT_PATH = PROTOCOL_ROOT / "Stage9_Input_Integrity_Commitment_v0.1.csv"
RUNTIME_STATE_PATH = RESULT_ROOT / "Stage9_Runtime_State_v0.1.json"
FINAL_RECORD_PATH = RESULT_ROOT / "Stage9_Hierarchical_DDO2_Specification_Freeze_Complete_v0.1.json"
OUTPUT_MANIFEST_PATH = RESULT_ROOT / "Stage9_Output_Integrity_Manifest_v0.1.csv"
REPORT_PATH = RESULT_ROOT / "Stage9_Hierarchical_DDO2_Specification_Freeze_Report_v0.1.md"
FIGURE_PATH = RESULT_ROOT / "Stage9_Evidence_And_Fit_Readiness_v0.1.png"

EXPECTED_STAGE7_FINAL_HASH = "293db1a6c41f86bcd4c94d91c2d6ad7dfdb5a9369fc4312beacce73b914cd4be"
EXPECTED_STAGE8_FINAL_HASH = "698c019b8516a84522831216b6cb0c85e047e6d0b401071db99e1d74ae6b2397"
EXPECTED_STAGE8B_FINAL_HASH = "4339402d843af177ef1181df1e3ae4fb7b0b6bd0211e250bb22cd9b255141024"
MAXIMUM_NEW_STAGE9_BYTES = 128 * 1024 * 1024

AXIS_SPECS = {
    "discrimination": {
        "outcome_column": "discrimination_pass",
        "failure_definition": "1 - discrimination_pass",
        "features": ["support_fraction", "fraction_beyond_source_q99", "atc_estimated_accuracy"],
    },
    "calibration": {
        "outcome_column": "calibration_pass",
        "failure_definition": "1 - calibration_pass",
        "features": ["atc_estimated_accuracy", "unlabeled_mixture_prevalence", "mean_entropy_nats"],
    },
    "operating_point": {
        "outcome_column": "operating_point_pass",
        "failure_definition": "1 - operating_point_pass",
        "features": ["unlabeled_mixture_prevalence", "target_to_source_logit_iqr_ratio", "mean_entropy_nats"],
    },
}
FROZEN_FEATURES = sorted({feature for spec in AXIS_SPECS.values() for feature in spec["features"]})
AUDIT_ONLY_COMPONENTS = [
    "domain_auc", "rbf_mmd2", "ot_cosine_cost", "mixture_wasserstein_residual_normalised"
]
OUTCOME_COLUMNS = [
    "discrimination_pass", "calibration_pass", "operating_point_pass",
    "target_auc", "target_ece10", "target_balanced_accuracy_at_0_5",
    "source_minus_target_auc", "calibration_ece_degradation", "calibration_brier_degradation",
]

FIT_GATES = {
    "minimum_total_eligible_edges": 30,
    "minimum_modalities": 4,
    "minimum_unique_datasets": 12,
    "minimum_failures_per_axis": 6,
    "minimum_passes_per_axis": 6,
    "minimum_modalities_per_axis_class": 2,
    "minimum_training_edges_per_grouped_fold": 18,
}

ANALYSIS_SPEC = {
    "scope": "SPECIFICATION_AND_VALIDATION_PROTOCOL_FREEZE_ONLY",
    "evidence_library": "Stage8B_ThreeModality_Eligible_Edge_Library_v0.1.csv",
    "unit_of_analysis": "directed within-task source-to-target dataset edge",
    "parent_final_record_sha256": EXPECTED_STAGE8B_FINAL_HASH,
    "outputs": ["discrimination_failure_risk", "calibration_failure_risk", "operating_point_failure_risk"],
    "single_global_score_authorised": False,
    "feature_selection_after_execution_authorised": False,
    "coefficient_fitting_authorised_only_if_all_fit_gates_pass": True,
    "axis_specs": AXIS_SPECS,
    "frozen_features": FROZEN_FEATURES,
    "audit_only_negative_or_descriptive_components": AUDIT_ONLY_COMPONENTS,
    "fit_gates": FIT_GATES,
    "model_family": "Bayesian hierarchical logistic model with shared slopes, modality-specific partially pooled slope deviations, and source/target dataset random intercepts",
    "primary_internal_validation": "leave-one-dataset-out; every edge incident to the held-out dataset is excluded from training",
    "secondary_internal_validation": "leave-one-modality-out; the complete modality is excluded from training",
    "future_blind_boundary": "acquisition roles sealed before labels; label-free predictions frozen before outcome unseal",
    "abstention": "source gate, missing input, task mismatch, feature OOD, or posterior uncertainty may force ABSTAIN",
    "future_data_accessed_in_stage9": False,
    "target_model_refit": False,
    "threshold_tuning": False,
    "final_external_validation": False,
}

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def sha256_json(payload):
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def atomic_json(path, payload):
    path = Path(path)
    temporary = Path(str(path) + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
    os.replace(temporary, path)

def canonical_csv_text(frame):
    return frame.to_csv(index=False, lineterminator="\n", float_format="%.12g")

def write_immutable_text(path, text):
    path = Path(path)
    if path.is_file():
        assert path.read_text(encoding="utf-8") == text, f"Existing immutable text differs: {path}"
    else:
        path.write_text(text, encoding="utf-8")

def write_immutable_csv(path, frame):
    write_immutable_text(path, canonical_csv_text(frame))

def write_immutable_json(path, payload):
    text = json.dumps(payload, indent=2, ensure_ascii=False) + "\n"
    write_immutable_text(path, text)

def verify_self_hashed_json(path, expected_hash=None, hash_field="final_record_sha256"):
    with Path(path).open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    claim = payload.get(hash_field)
    assert claim, f"Missing {hash_field}: {path}"
    without_claim = dict(payload)
    without_claim.pop(hash_field)
    assert sha256_json(without_claim) == claim, f"Self-hash mismatch: {path}"
    if expected_hash is not None:
        assert claim == expected_hash, f"Unexpected frozen parent hash: {path}"
    return payload, claim

def verify_integrity_manifest(root, manifest_path):
    manifest = pd.read_csv(manifest_path)
    assert list(manifest.columns) == ["relative_path", "size_bytes", "sha256"]
    for record in manifest.itertuples(index=False):
        path = Path(root) / record.relative_path
        assert path.is_file(), f"Missing parent output: {path}"
        assert path.stat().st_size == int(record.size_bytes), f"Parent size mismatch: {path}"
        assert sha256_file(path) == record.sha256, f"Parent hash mismatch: {path}"
    return manifest

def normalised_notebook_source_sha256(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        notebook = json.load(handle)
    payload = []
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") not in {"code", "markdown"}:
            continue
        source = cell.get("source", [])
        source = "".join(source) if isinstance(source, list) else str(source)
        payload.append({"cell_type": cell["cell_type"], "source": source.replace("\r\n", "\n")})
    return sha256_json(payload)

required_inputs = [
    NOTEBOOK_PATH, STAGE7_FINAL_PATH, STAGE7_CANDIDATE_PATH, STAGE8_FINAL_PATH,
    STAGE8B_FINAL_PATH, STAGE8B_MANIFEST_PATH, EDGE_LIBRARY_PATH,
]
assert all(path.is_file() for path in required_inputs), "A frozen input is missing; do not substitute a similarly named file."

stage7_final, stage7_claim = verify_self_hashed_json(STAGE7_FINAL_PATH, EXPECTED_STAGE7_FINAL_HASH)
stage8_final, stage8_claim = verify_self_hashed_json(STAGE8_FINAL_PATH, EXPECTED_STAGE8_FINAL_HASH)
stage8b_final, stage8b_claim = verify_self_hashed_json(STAGE8B_FINAL_PATH, EXPECTED_STAGE8B_FINAL_HASH)
stage8b_integrity = verify_integrity_manifest(STAGE8B_ROOT, STAGE8B_MANIFEST_PATH)

input_paths = [
    STAGE7_FINAL_PATH, STAGE7_CANDIDATE_PATH, STAGE8_FINAL_PATH,
    STAGE8B_FINAL_PATH, STAGE8B_MANIFEST_PATH, EDGE_LIBRARY_PATH,
]
input_commitment = pd.DataFrame([
    {
        "role": path.name,
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in input_paths
])
write_immutable_csv(INPUT_COMMITMENT_PATH, input_commitment)

notebook_source_hash = normalised_notebook_source_sha256(NOTEBOOK_PATH)
if PROTOCOL_SEAL_PATH.is_file():
    with PROTOCOL_SEAL_PATH.open("r", encoding="utf-8") as handle:
        protocol_seal = json.load(handle)
    seal_claim = protocol_seal["seal_sha256"]
    seal_without_claim = dict(protocol_seal)
    seal_without_claim.pop("seal_sha256")
    assert sha256_json(seal_without_claim) == seal_claim
    assert protocol_seal["analysis_spec"] == ANALYSIS_SPEC
    assert protocol_seal["notebook_source_sha256"] == notebook_source_hash
    assert protocol_seal["parent_stage8b_final_record_sha256"] == EXPECTED_STAGE8B_FINAL_HASH
else:
    protocol_seal = {
        "stage": "Stage9",
        "decision": "HIERARCHICAL_DDO2_SPECIFICATION_PROTOCOL_SEALED",
        "parent_stage8b_final_record_sha256": EXPECTED_STAGE8B_FINAL_HASH,
        "notebook_source_sha256": notebook_source_hash,
        "input_commitment_sha256": sha256_file(INPUT_COMMITMENT_PATH),
        "analysis_spec": ANALYSIS_SPEC,
        "sealed_utc": utc_now(),
    }
    protocol_seal["seal_sha256"] = sha256_json(protocol_seal)
    write_immutable_json(PROTOCOL_SEAL_PATH, protocol_seal)
seal_claim = protocol_seal["seal_sha256"]

runtime_state = {
    "stage": "Stage9", "protocol_sealed": True, "parent_lineage_verified": True,
    "eligible_edge_library_loaded": False, "future_data_accessed": False,
    "final_ddo2_fitted": False, "final_blind_validation_performed": False,
    "last_updated_utc": utc_now(),
}
atomic_json(RUNTIME_STATE_PATH, runtime_state)

print("Stage 7 final record verified:", stage7_claim)
print("Stage 8 final record verified:", stage8_claim)
print("Stage 8B final record verified:", stage8b_claim)
print("Stage 8B integrity rows verified:", len(stage8b_integrity))
print("Stage 9 protocol seal:", seal_claim)
print("Future data accessed / final DDO2 fitted:", False, "/", False)


Mounted at /content/drive
================ STAGE 9 HIERARCHICAL DDO2 SPECIFICATION PREFLIGHT ================
Stage 7 final record verified: 293db1a6c41f86bcd4c94d91c2d6ad7dfdb5a9369fc4312beacce73b914cd4be
Stage 8 final record verified: 698c019b8516a84522831216b6cb0c85e047e6d0b401071db99e1d74ae6b2397
Stage 8B final record verified: 4339402d843af177ef1181df1e3ae4fb7b0b6bd0211e250bb22cd9b255141024
Stage 8B integrity rows verified: 24
Stage 9 protocol seal: f5689995ec6ab0dcf0bcacf813d578edd47edbedf8fb96b7809ec674e3cad593
Future data accessed / final DDO2 fitted: False / False


In [2]:
# @title 09-1. Load and validate the sealed 21-edge evidence library
def coerce_bool(series, name):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    converted = series.astype(str).str.strip().str.lower().map(mapping)
    assert converted.notna().all(), f"Unparseable boolean values in {name}"
    return converted.astype(bool)

edges = pd.read_csv(EDGE_LIBRARY_PATH)
for column in ["source_recoverable", "discrimination_pass", "calibration_pass", "operating_point_pass"]:
    edges[column] = coerce_bool(edges[column], column)

required_columns = set(
    ["edge_id", "modality", "task", "source", "target", "origin_stage", "source_recoverable"]
    + FROZEN_FEATURES + AUDIT_ONLY_COMPONENTS + OUTCOME_COLUMNS
)
quality_checks = []

def add_check(name, passed, observed, expected):
    quality_checks.append({"check": name, "passed": bool(passed), "observed": str(observed), "expected": str(expected)})

add_check("row_count", len(edges) == 21, len(edges), 21)
add_check("required_columns_present", required_columns.issubset(edges.columns), len(required_columns & set(edges.columns)), len(required_columns))
add_check("unique_edge_id", edges["edge_id"].is_unique, edges["edge_id"].nunique(), len(edges))
add_check("no_self_edges", (edges["source"] != edges["target"]).all(), int((edges["source"] == edges["target"]).sum()), 0)
add_check("all_sources_recoverable", edges["source_recoverable"].all(), int(edges["source_recoverable"].sum()), len(edges))
add_check("three_modalities", edges["modality"].nunique() == 3, edges["modality"].nunique(), 3)
all_datasets = sorted(set(edges["source"]) | set(edges["target"]))
add_check("ten_unique_datasets", len(all_datasets) == 10, len(all_datasets), 10)
add_check("one_task_per_modality", (edges.groupby("modality")["task"].nunique() == 1).all(), edges.groupby("modality")["task"].nunique().to_dict(), "all 1")
expected_origin = {"Stage7": 9, "Stage8": 6, "Stage8B": 6}
add_check("frozen_origin_counts", edges["origin_stage"].value_counts().to_dict() == expected_origin, edges["origin_stage"].value_counts().to_dict(), expected_origin)
add_check("no_feature_outcome_leakage", set(FROZEN_FEATURES).isdisjoint(OUTCOME_COLUMNS), set(FROZEN_FEATURES) & set(OUTCOME_COLUMNS), "empty")

for column in FROZEN_FEATURES + AUDIT_ONLY_COMPONENTS:
    values = pd.to_numeric(edges[column], errors="coerce")
    add_check(f"finite::{column}", np.isfinite(values).all(), int(np.isfinite(values).sum()), len(edges))

data_quality_audit = pd.DataFrame(quality_checks)
write_immutable_csv(AUDIT_ROOT / "Stage9_Edge_Library_Data_Quality_Audit_v0.1.csv", data_quality_audit)
if not data_quality_audit["passed"].all():
    print(data_quality_audit.loc[~data_quality_audit["passed"]].to_string(index=False))
    raise AssertionError("Stage 9 data-quality gate failed; no specification artifact may be promoted.")

edges = edges.sort_values(["modality", "source", "target"]).reset_index(drop=True)
edges["discrimination_failure"] = (~edges["discrimination_pass"]).astype(int)
edges["calibration_failure"] = (~edges["calibration_pass"]).astype(int)
edges["operating_point_failure"] = (~edges["operating_point_pass"]).astype(int)

runtime_state.update({
    "eligible_edge_library_loaded": True,
    "eligible_edges": int(len(edges)),
    "modalities": int(edges["modality"].nunique()),
    "unique_datasets": int(len(all_datasets)),
    "last_updated_utc": utc_now(),
})
atomic_json(RUNTIME_STATE_PATH, runtime_state)

print("Eligible directed edges:", len(edges))
print("Modalities:", sorted(edges["modality"].unique()))
print("Datasets:", all_datasets)
print("Origin counts:", edges["origin_stage"].value_counts().to_dict())
print("All data-quality and leakage checks passed.")


Eligible directed edges: 21
Modalities: ['chest_radiography', 'dermoscopy', 'retinal_fundus']
Datasets: ['APTOS_2019', 'DeepDRiD', 'EyePACS_2015', 'HAM10000', 'IDRiD', 'ISIC_MSK1', 'ISIC_UDA1', 'Montgomery_CXR', 'Shenzhen_CXR', 'TBX11K']
Origin counts: {'Stage7': 9, 'Stage8B': 6, 'Stage8': 6}
All data-quality and leakage checks passed.


In [3]:
# @title 09-2. Audit failure axes, edge dependence, and cross-modality heterogeneity
from scipy.stats import spearmanr

axis_columns = {
    "discrimination": "discrimination_failure",
    "calibration": "calibration_failure",
    "operating_point": "operating_point_failure",
}

summary_rows = []
for modality, group in edges.groupby("modality", sort=True):
    row = {
        "modality": modality,
        "eligible_edges": len(group),
        "unique_datasets": len(set(group["source"]) | set(group["target"])),
    }
    for axis, failure_column in axis_columns.items():
        row[f"{axis}_failures"] = int(group[failure_column].sum())
        row[f"{axis}_passes"] = int(len(group) - group[failure_column].sum())
    summary_rows.append(row)
modality_summary = pd.DataFrame(summary_rows)
write_immutable_csv(AUDIT_ROOT / "Stage9_Modality_Axis_Outcome_Summary_v0.1.csv", modality_summary)

incidence_rows = []
for dataset in all_datasets:
    source_mask = edges["source"].eq(dataset)
    target_mask = edges["target"].eq(dataset)
    incident_mask = source_mask | target_mask
    modalities = sorted(edges.loc[incident_mask, "modality"].unique())
    incidence_rows.append({
        "dataset": dataset,
        "modality": "|".join(modalities),
        "as_source_edges": int(source_mask.sum()),
        "as_target_edges": int(target_mask.sum()),
        "incident_edges": int(incident_mask.sum()),
        "non_independence_warning": "ALL_INCIDENT_EDGES_SHARE_A_DATASET",
    })
dataset_incidence = pd.DataFrame(incidence_rows)
write_immutable_csv(AUDIT_ROOT / "Stage9_Dataset_Edge_Incidence_Audit_v0.1.csv", dataset_incidence)

feature_quality_rows = []
for column in FROZEN_FEATURES + AUDIT_ONLY_COMPONENTS:
    values = pd.to_numeric(edges[column], errors="coerce")
    feature_quality_rows.append({
        "component": column,
        "role": "FROZEN_MODEL_INPUT" if column in FROZEN_FEATURES else "AUDIT_ONLY_NOT_MODEL_INPUT",
        "nonmissing": int(values.notna().sum()),
        "unique_values": int(values.nunique(dropna=True)),
        "minimum": float(values.min()),
        "median": float(values.median()),
        "maximum": float(values.max()),
    })
feature_quality = pd.DataFrame(feature_quality_rows)
write_immutable_csv(AUDIT_ROOT / "Stage9_LabelFree_Component_Quality_Audit_v0.1.csv", feature_quality)

joint_state = (
    edges.assign(
        axis_state=edges.apply(
            lambda row: f"D{int(row.discrimination_failure)}_C{int(row.calibration_failure)}_O{int(row.operating_point_failure)}",
            axis=1,
        )
    )
    .groupby(["modality", "axis_state"], as_index=False)
    .size()
    .rename(columns={"size": "edges"})
)
write_immutable_csv(AUDIT_ROOT / "Stage9_Joint_Failure_State_Audit_v0.1.csv", joint_state)

association_rows = []
association_components = FROZEN_FEATURES + AUDIT_ONLY_COMPONENTS
for scope, group in [("ALL_MODALITIES", edges)] + list(edges.groupby("modality", sort=True)):
    for axis, failure_column in axis_columns.items():
        for component in association_components:
            x = pd.to_numeric(group[component], errors="coerce")
            y = group[failure_column].astype(float)
            valid = x.notna() & y.notna()
            if valid.sum() >= 4 and x[valid].nunique() >= 2 and y[valid].nunique() >= 2:
                rho, pvalue = spearmanr(x[valid], y[valid])
            else:
                rho, pvalue = np.nan, np.nan
            association_rows.append({
                "scope": scope,
                "axis": axis,
                "component": component,
                "n_edges": int(valid.sum()),
                "spearman_rho_with_failure": rho,
                "nominal_p_value_not_for_inference": pvalue,
                "status": "DESCRIPTIVE_ONLY_NOT_USED_FOR_FEATURE_SELECTION",
            })
association_audit = pd.DataFrame(association_rows)
write_immutable_csv(AUDIT_ROOT / "Stage9_Component_Failure_Heterogeneity_Audit_v0.1.csv", association_audit)

axis_dependence_rows = []
axis_items = list(axis_columns.items())
for index, (axis_a, col_a) in enumerate(axis_items):
    for axis_b, col_b in axis_items[index + 1:]:
        correlation = np.corrcoef(edges[col_a].astype(float), edges[col_b].astype(float))[0, 1]
        axis_dependence_rows.append({
            "axis_a": axis_a, "axis_b": axis_b, "phi_correlation": float(correlation),
            "interpretation": "DESCRIPTIVE_DEPENDENCE_DO_NOT_COLLAPSE_TO_ONE_SCORE",
        })
axis_dependence = pd.DataFrame(axis_dependence_rows)
write_immutable_csv(AUDIT_ROOT / "Stage9_Failure_Axis_Dependence_Audit_v0.1.csv", axis_dependence)

print("Modality-level axis counts")
print(modality_summary.to_string(index=False))
print("\nDataset incidence confirms that 21 rows are not 21 independent experiments.")
print("Maximum incident edges for one dataset:", int(dataset_incidence["incident_edges"].max()))
print("Descriptive associations were saved but were not used to alter the frozen feature sets.")


Modality-level axis counts
         modality  eligible_edges  unique_datasets  discrimination_failures  discrimination_passes  calibration_failures  calibration_passes  operating_point_failures  operating_point_passes
chest_radiography               6                3                        3                      3                     6                   0                         6                       0
       dermoscopy               6                3                        4                      2                     4                   2                         6                       0
   retinal_fundus               9                4                        4                      5                     8                   1                         6                       3

Dataset incidence confirms that 21 rows are not 21 independent experiments.
Maximum incident edges for one dataset: 5
Descriptive associations were saved but were not used to alter the frozen feature sets.


In [4]:
# @title 09-3. Materialise the frozen hierarchical DDO2 specification and ABSTAIN contract
input_schema_rows = [
    {
        "component": "support_fraction",
        "semantic_group": "representation_support",
        "transform_before_training": "logit(clip(x,1e-4,1-1e-4)) then training-only robust scaling",
        "discrimination": True, "calibration": False, "operating_point": False,
        "target_labels_required": False,
    },
    {
        "component": "fraction_beyond_source_q99",
        "semantic_group": "representation_support",
        "transform_before_training": "logit(clip(x,1e-4,1-1e-4)) then training-only robust scaling",
        "discrimination": True, "calibration": False, "operating_point": False,
        "target_labels_required": False,
    },
    {
        "component": "atc_estimated_accuracy",
        "semantic_group": "confidence_consistency",
        "transform_before_training": "logit(clip(x,1e-4,1-1e-4)) then training-only robust scaling",
        "discrimination": True, "calibration": True, "operating_point": False,
        "target_labels_required": False,
    },
    {
        "component": "unlabeled_mixture_prevalence",
        "semantic_group": "score_mixture",
        "transform_before_training": "logit(clip(x,1e-4,1-1e-4)) then training-only robust scaling",
        "discrimination": False, "calibration": True, "operating_point": True,
        "target_labels_required": False,
    },
    {
        "component": "mean_entropy_nats",
        "semantic_group": "score_uncertainty",
        "transform_before_training": "training-only robust scaling",
        "discrimination": False, "calibration": True, "operating_point": True,
        "target_labels_required": False,
    },
    {
        "component": "target_to_source_logit_iqr_ratio",
        "semantic_group": "score_spread",
        "transform_before_training": "log(clip(x,1e-6,None)) then training-only robust scaling",
        "discrimination": False, "calibration": False, "operating_point": True,
        "target_labels_required": False,
    },
]
input_schema = pd.DataFrame(input_schema_rows)
assert set(input_schema["component"]) == set(FROZEN_FEATURES)
write_immutable_csv(SPEC_ROOT / "Stage9_Frozen_DDO2_LabelFree_Input_Schema_v0.1.csv", input_schema)

outcome_axis_definitions = pd.DataFrame([
    {
        "axis": "discrimination",
        "development_label": "1 - discrimination_pass",
        "parent_definition": "pass only under the frozen source/target AUC and lower-CI rule used in Stages 7-8B",
        "prediction": "posterior probability of discrimination failure",
        "independent_output": True,
    },
    {
        "axis": "calibration",
        "development_label": "1 - calibration_pass",
        "parent_definition": "pass only under the frozen ECE/Brier degradation tolerance used in Stages 7-8B",
        "prediction": "posterior probability of calibration failure",
        "independent_output": True,
    },
    {
        "axis": "operating_point",
        "development_label": "1 - operating_point_pass",
        "parent_definition": "pass only under the frozen threshold-0.5 balanced-accuracy rule used in Stages 7-8B",
        "prediction": "posterior probability of fixed-operating-point failure",
        "independent_output": True,
    },
])
write_immutable_csv(SPEC_ROOT / "Stage9_Frozen_DDO2_Outcome_Axis_Definitions_v0.1.csv", outcome_axis_definitions)

model_specification = {
    "name": "Hierarchical_DDO2_v0.1_UNFITTED_SPECIFICATION",
    "status": "SPECIFICATION_FROZEN_COEFFICIENTS_NOT_YET_ESTIMATED",
    "outputs": list(AXIS_SPECS),
    "one_model_per_axis": True,
    "collapsed_global_score": False,
    "axis_features": {axis: spec["features"] for axis, spec in AXIS_SPECS.items()},
    "preprocessing": {
        "fit_boundary": "all transforms and robust scaling fitted on the training fold only",
        "robust_center": "training median",
        "robust_scale": "1.4826*MAD; if zero use IQR/1.349; if still zero use 1",
        "post_scale_clip": [-5.0, 5.0],
        "missing_value_imputation": "not authorised; missing required input causes ABSTAIN",
    },
    "linear_predictor": "eta[a,e]=alpha[a]+(beta[a]+v[a,modality[e]])^T z[a,e]+u[a,modality[e]]+s[a,source[e]]+t[a,target[e]]",
    "likelihood": "Bernoulli(logit^-1(eta[a,e]))",
    "priors": {
        "alpha": "Normal(0,1.5)",
        "shared_beta": "Normal(0,0.75)",
        "modality_intercept_sd": "HalfNormal(0.5)",
        "modality_slope_deviation_sd": "HalfNormal(0.35)",
        "source_dataset_intercept_sd": "HalfNormal(0.35)",
        "target_dataset_intercept_sd": "HalfNormal(0.35)",
    },
    "unseen_modality": "set modality effects to population mean zero and propagate between-modality variance; do not silently map to an existing modality",
    "unseen_dataset": "set source/target random intercept to population mean zero and propagate dataset-level variance",
    "fit_engine": "Bayesian sampler with 4 chains, >=1000 warmup and >=1000 retained draws per chain",
    "convergence_gate": "R-hat <=1.01, bulk/tail ESS >=400, zero unresolved divergences; otherwise no deployable fit",
    "fit_gates": FIT_GATES,
    "validation": {
        "primary": "leave-one-dataset-out with all incident edges held out",
        "secondary": "leave-one-modality-out",
        "no_edge_row_random_split": True,
        "primary_metrics": ["Brier score", "AUROC when both classes exist", "log loss", "90% interval coverage", "abstention coverage"],
        "uncertainty": "hierarchical cluster bootstrap or posterior predictive intervals; never naive edge-row bootstrap",
    },
    "forbidden_after_freeze": [
        "outcome-driven feature replacement", "sign reversal chosen after blind labels", "target refit",
        "target threshold tuning", "modality reclassification after results", "reporting only non-abstained successes",
    ],
}
model_specification["specification_sha256"] = sha256_json(model_specification)
MODEL_SPEC_HASH = model_specification["specification_sha256"]
write_immutable_json(SPEC_ROOT / "Stage9_Frozen_Hierarchical_DDO2_Model_Specification_v0.1.json", model_specification)

abstain_policy = {
    "policy": "AXIS_SPECIFIC_ABSTENTION_WITHOUT_SINGLE_GLOBAL_SCORE",
    "source_gate": "ABSTAIN if the frozen source recoverability gate fails",
    "task_gate": "ABSTAIN if source and target endpoints or unit definitions are not harmonised before prediction",
    "missingness_gate": "ABSTAIN on any missing or non-finite required axis input",
    "feature_ood_gate": "ABSTAIN if any pre-clipped training-scale |z| > 5 or transformed-input kNN distance exceeds the training 99th percentile",
    "posterior_uncertainty_gate": "ABSTAIN for that axis if the 90% posterior interval width exceeds 0.50",
    "action_thresholds": {
        "LOW_FAILURE_RISK": "90% posterior upper bound <=0.33",
        "HIGH_FAILURE_RISK": "90% posterior lower bound >=0.67",
        "ABSTAIN": "all other intervals",
    },
    "unseen_modality": "use shared population effect with full hierarchical uncertainty; unseen status alone is not automatic abstention",
    "reporting": "always report per-axis probability, 90% interval, action, abstention reason, and total coverage",
}
abstain_policy["policy_sha256"] = sha256_json(abstain_policy)
ABSTAIN_POLICY_HASH = abstain_policy["policy_sha256"]
write_immutable_json(SPEC_ROOT / "Stage9_Frozen_DDO2_ABSTAIN_Policy_v0.1.json", abstain_policy)

excluded_components = pd.DataFrame([
    {"component": component, "status": "AUDIT_ONLY_EXCLUDED_FROM_V0.1_MODEL", "reason": reason}
    for component, reason in {
        "domain_auc": "generic domain separability is a non-specific negative control",
        "rbf_mmd2": "not selected in the pre-sealed parsimonious axis sets",
        "ot_cosine_cost": "not selected in the pre-sealed parsimonious axis sets",
        "mixture_wasserstein_residual_normalised": "retained for descriptive audit, not outcome-driven feature expansion",
        "target_mean_knn_distance": "raw distance is not commensurate across frozen modality encoders",
        "target_mean_knn_distance_normalised": "missing for retinal Stage 7 edges; no post-hoc cross-modality imputation authorised",
    }.items()
])
write_immutable_csv(SPEC_ROOT / "Stage9_Frozen_Excluded_Component_Register_v0.1.csv", excluded_components)

print("Frozen model specification hash:", MODEL_SPEC_HASH)
print("Frozen ABSTAIN policy hash:", ABSTAIN_POLICY_HASH)
print("Outputs remain separate:", list(AXIS_SPECS))
print("Fitted coefficients produced in Stage 9:", False)


Frozen model specification hash: e2a0d8f65143e0b9d58d740595a39b803f1684481c4f8af01f05935bfaacb732
Frozen ABSTAIN policy hash: a7d744dfdec1b4f0847d2e6b22a9e6368738630d3b96b3328584ce0ebf1da9b0
Outputs remain separate: ['discrimination', 'calibration', 'operating_point']
Fitted coefficients produced in Stage 9: False


In [5]:
# @title 09-4. Register leakage-safe folds and enforce coefficient-fit eligibility gates
fold_rows = []
for dataset in all_datasets:
    test_mask = edges["source"].eq(dataset) | edges["target"].eq(dataset)
    train = edges.loc[~test_mask]
    test = edges.loc[test_mask]
    leakage = train["source"].eq(dataset).any() or train["target"].eq(dataset).any()
    row = {
        "scheme": "LEAVE_ONE_DATASET_OUT",
        "held_out_group": dataset,
        "n_train_edges": len(train),
        "n_test_edges": len(test),
        "group_leakage_detected": bool(leakage),
    }
    for axis, failure_column in axis_columns.items():
        row[f"train_{axis}_failures"] = int(train[failure_column].sum())
        row[f"train_{axis}_passes"] = int(len(train) - train[failure_column].sum())
        row[f"test_{axis}_failures"] = int(test[failure_column].sum())
        row[f"test_{axis}_passes"] = int(len(test) - test[failure_column].sum())
    row["fold_meets_minimum_training_edges"] = len(train) >= FIT_GATES["minimum_training_edges_per_grouped_fold"]
    fold_rows.append(row)

for modality in sorted(edges["modality"].unique()):
    test_mask = edges["modality"].eq(modality)
    train = edges.loc[~test_mask]
    test = edges.loc[test_mask]
    leakage = train["modality"].eq(modality).any()
    row = {
        "scheme": "LEAVE_ONE_MODALITY_OUT",
        "held_out_group": modality,
        "n_train_edges": len(train),
        "n_test_edges": len(test),
        "group_leakage_detected": bool(leakage),
    }
    for axis, failure_column in axis_columns.items():
        row[f"train_{axis}_failures"] = int(train[failure_column].sum())
        row[f"train_{axis}_passes"] = int(len(train) - train[failure_column].sum())
        row[f"test_{axis}_failures"] = int(test[failure_column].sum())
        row[f"test_{axis}_passes"] = int(len(test) - test[failure_column].sum())
    row["fold_meets_minimum_training_edges"] = len(train) >= FIT_GATES["minimum_training_edges_per_grouped_fold"]
    fold_rows.append(row)

fold_registry = pd.DataFrame(fold_rows)
assert not fold_registry["group_leakage_detected"].any()
write_immutable_csv(VALIDATION_ROOT / "Stage9_Frozen_Grouped_Validation_Fold_Registry_v0.1.csv", fold_registry)

eligibility_rows = []
def eligibility_gate(gate, observed, required, passed, scope="GLOBAL"):
    eligibility_rows.append({
        "scope": scope, "gate": gate, "observed": observed, "required": required,
        "passed": bool(passed), "consequence_if_failed": "COEFFICIENT_FIT_NOT_AUTHORISED",
    })

eligibility_gate("total_eligible_edges", len(edges), FIT_GATES["minimum_total_eligible_edges"], len(edges) >= FIT_GATES["minimum_total_eligible_edges"])
eligibility_gate("modalities", edges["modality"].nunique(), FIT_GATES["minimum_modalities"], edges["modality"].nunique() >= FIT_GATES["minimum_modalities"])
eligibility_gate("unique_datasets", len(all_datasets), FIT_GATES["minimum_unique_datasets"], len(all_datasets) >= FIT_GATES["minimum_unique_datasets"])
eligibility_gate(
    "all_grouped_folds_minimum_training_edges",
    int(fold_registry["fold_meets_minimum_training_edges"].sum()), len(fold_registry),
    fold_registry["fold_meets_minimum_training_edges"].all(),
)

for axis, failure_column in axis_columns.items():
    failures = int(edges[failure_column].sum())
    passes = int(len(edges) - failures)
    failure_modalities = int(edges.loc[edges[failure_column].eq(1), "modality"].nunique())
    pass_modalities = int(edges.loc[edges[failure_column].eq(0), "modality"].nunique())
    eligibility_gate("minimum_failures", failures, FIT_GATES["minimum_failures_per_axis"], failures >= FIT_GATES["minimum_failures_per_axis"], axis)
    eligibility_gate("minimum_passes", passes, FIT_GATES["minimum_passes_per_axis"], passes >= FIT_GATES["minimum_passes_per_axis"], axis)
    eligibility_gate(
        "failure_class_modalities", failure_modalities, FIT_GATES["minimum_modalities_per_axis_class"],
        failure_modalities >= FIT_GATES["minimum_modalities_per_axis_class"], axis,
    )
    eligibility_gate(
        "pass_class_modalities", pass_modalities, FIT_GATES["minimum_modalities_per_axis_class"],
        pass_modalities >= FIT_GATES["minimum_modalities_per_axis_class"], axis,
    )

fit_eligibility = pd.DataFrame(eligibility_rows)
FIT_AUTHORISED = bool(fit_eligibility["passed"].all())
write_immutable_csv(VALIDATION_ROOT / "Stage9_DDO2_Coefficient_Fit_Eligibility_Audit_v0.1.csv", fit_eligibility)

if FIT_AUTHORISED:
    raise AssertionError(
        "Unexpectedly all fit gates passed despite the frozen 21-edge input. Stage 9 is specification-only; "
        "stop and create a separately sealed coefficient-fit stage."
    )

unfitted_model_state = {
    "model": "Hierarchical_DDO2_v0.1",
    "model_specification_sha256": MODEL_SPEC_HASH,
    "abstain_policy_sha256": ABSTAIN_POLICY_HASH,
    "status": "UNFITTED_BY_FROZEN_PROTOCOL",
    "fit_authorised": False,
    "coefficients": None,
    "scaler_parameters": None,
    "failed_gates": fit_eligibility.loc[~fit_eligibility["passed"], ["scope", "gate", "observed", "required"]].to_dict("records"),
    "interpretation": "The existing evidence freezes the method family but is not an external validation set and is insufficient for a deployable three-axis hierarchical fit.",
}
unfitted_model_state["state_sha256"] = sha256_json(unfitted_model_state)
UNFITTED_STATE_HASH = unfitted_model_state["state_sha256"]
write_immutable_json(SPEC_ROOT / "Stage9_Frozen_Unfitted_DDO2_State_v0.1.json", unfitted_model_state)

print("Coefficient fit authorised:", FIT_AUTHORISED)
print("Failed gates")
print(fit_eligibility.loc[~fit_eligibility["passed"], ["scope", "gate", "observed", "required"]].to_string(index=False))
print("Unfitted model-state hash:", UNFITTED_STATE_HASH)


Coefficient fit authorised: False
Failed gates
          scope                                     gate  observed  required
         GLOBAL                     total_eligible_edges        21        30
         GLOBAL                               modalities         3         4
         GLOBAL                          unique_datasets        10        12
         GLOBAL all_grouped_folds_minimum_training_edges         1        13
    calibration                           minimum_passes         3         6
operating_point                           minimum_passes         3         6
operating_point                    pass_class_modalities         1         2
Unfitted model-state hash: e7f119fa779db67274ea4e4b4a70f4aee943d0fbe4f9eb34e587abc087dc26d8


In [6]:
# @title 09-5. Freeze the acquisition-role boundary and future blind-validation interface
axis_shortfalls = []
for axis, failure_column in axis_columns.items():
    failures = int(edges[failure_column].sum())
    passes = int(len(edges) - failures)
    axis_shortfalls.append({
        "axis": axis,
        "current_failures": failures,
        "current_passes": passes,
        "additional_failures_needed_for_fit_gate": max(0, FIT_GATES["minimum_failures_per_axis"] - failures),
        "additional_passes_needed_for_fit_gate": max(0, FIT_GATES["minimum_passes_per_axis"] - passes),
    })
axis_shortfall_table = pd.DataFrame(axis_shortfalls)
write_immutable_csv(BLIND_ROOT / "Stage9_Axis_Class_Balance_Shortfall_v0.1.csv", axis_shortfall_table)

acquisition_role_template = pd.DataFrame([
    {
        "dataset_id": "TBD_DEVELOPMENT_EXTENSION",
        "modality": "TBD",
        "task": "TBD_HARMONISED_BINARY_ENDPOINT",
        "role_frozen_before_label_access": "DEVELOPMENT_EXTENSION",
        "labels_permitted_before_label_free_prediction_freeze": True,
        "eligible_for_model_or_scaler_fit": True,
        "eligible_for_final_blind_claim": False,
        "required_action": "add domains until every frozen fit gate passes",
    },
    {
        "dataset_id": "TBD_LOCKED_BLIND_TARGET",
        "modality": "TBD",
        "task": "TBD_HARMONISED_BINARY_ENDPOINT",
        "role_frozen_before_label_access": "LOCKED_BLIND_TEST",
        "labels_permitted_before_label_free_prediction_freeze": False,
        "eligible_for_model_or_scaler_fit": False,
        "eligible_for_final_blind_claim": True,
        "required_action": "reserve at least two full datasets or one complete modality; freeze predictions before labels",
    },
])
write_immutable_csv(BLIND_ROOT / "Stage9_Stage10_Acquisition_Role_Registry_Template_v0.1.csv", acquisition_role_template)

future_input_columns = [
    "edge_id", "modality", "task", "source", "target", "source_recoverable",
] + FROZEN_FEATURES + [
    "input_manifest_sha256", "source_axis_sha256", "label_free_prediction_frozen_utc",
]
future_input_template = pd.DataFrame(columns=future_input_columns)
write_immutable_csv(BLIND_ROOT / "Stage9_Future_Blind_Edge_LabelFree_Input_Template_v0.1.csv", future_input_template)

future_prediction_columns = [
    "edge_id", "axis", "failure_probability_posterior_mean", "failure_probability_q05",
    "failure_probability_q95", "action", "abstain_reason", "model_state_sha256",
    "input_row_sha256", "prediction_frozen_utc",
]
future_prediction_template = pd.DataFrame(columns=future_prediction_columns)
write_immutable_csv(BLIND_ROOT / "Stage9_Future_Blind_DDO2_Prediction_Freeze_Template_v0.1.csv", future_prediction_template)

future_blind_protocol = {
    "name": "Future_DDO2_Development_Extension_And_Blind_Validation_Protocol_v0.1",
    "parent_stage9_protocol_seal_sha256": seal_claim,
    "model_specification_sha256": MODEL_SPEC_HASH,
    "abstain_policy_sha256": ABSTAIN_POLICY_HASH,
    "phase_1_role_seal": "assign every new dataset to DEVELOPMENT_EXTENSION or LOCKED_BLIND_TEST before outcome analysis",
    "phase_2_development_extension": "only development-extension labels may be used; stop unless every Stage 9 fit gate passes",
    "phase_3_model_freeze": "fit exactly the frozen model family; freeze transforms, coefficients, posterior draws, software versions, and model-state hash",
    "phase_4_blind_prediction": "compute label-free components and freeze all three axis predictions and abstentions for locked-blind edges",
    "phase_5_one_time_unseal": "retrieve locked-blind outcomes once, evaluate all frozen predictions including abstentions, and publish every edge",
    "minimum_blind_reserve": "at least two complete datasets or one complete modality not used for feature choice, preprocessing, priors, coefficients, or thresholds",
    "primary_evaluation_unit": "dataset/modality groups, not independent edge rows",
    "prohibited": [
        "moving a failed blind dataset into development", "post-unseal sign changes", "threshold tuning",
        "target refit", "dropping abstained or failed edges", "calling internal grouped CV external validation",
    ],
}
future_blind_protocol["protocol_sha256"] = sha256_json(future_blind_protocol)
FUTURE_BLIND_PROTOCOL_HASH = future_blind_protocol["protocol_sha256"]
write_immutable_json(BLIND_ROOT / "Stage9_Frozen_Future_Blind_Validation_Protocol_v0.1.json", future_blind_protocol)

next_stage_requirements = pd.DataFrame([
    {"requirement": "total eligible directed edges", "current": len(edges), "minimum_before_fit": FIT_GATES["minimum_total_eligible_edges"], "met": len(edges) >= FIT_GATES["minimum_total_eligible_edges"]},
    {"requirement": "modalities represented in development", "current": edges["modality"].nunique(), "minimum_before_fit": FIT_GATES["minimum_modalities"], "met": edges["modality"].nunique() >= FIT_GATES["minimum_modalities"]},
    {"requirement": "unique datasets represented in development", "current": len(all_datasets), "minimum_before_fit": FIT_GATES["minimum_unique_datasets"], "met": len(all_datasets) >= FIT_GATES["minimum_unique_datasets"]},
    {"requirement": "locked blind reserve", "current": 0, "minimum_before_fit": 2, "met": False},
])
write_immutable_csv(BLIND_ROOT / "Stage9_Next_Stage_Data_Requirement_Summary_v0.1.csv", next_stage_requirements)

print("Future blind protocol hash:", FUTURE_BLIND_PROTOCOL_HASH)
print("Development and locked-blind roles are mutually exclusive.")
print("Locked blind labels may not be accessed before three-axis predictions and abstentions are frozen.")


Future blind protocol hash: acef1a8da90b95e875ed29572ca4a8283a5035ebfc1a1bf9fc787c487a01ec5a
Development and locked-blind roles are mutually exclusive.
Locked blind labels may not be accessed before three-axis predictions and abstentions are frozen.


In [7]:
# @title 09-6. Create the decision record, integrity manifest, and handoff report
import matplotlib.pyplot as plt

if not FIGURE_PATH.is_file():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    x = np.arange(len(modality_summary))
    width = 0.24
    for offset, axis in zip([-width, 0, width], ["discrimination", "calibration", "operating_point"]):
        axes[0].bar(
            x + offset,
            modality_summary[f"{axis}_failures"],
            width=width,
            label=axis.replace("_", " ").title(),
        )
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(modality_summary["modality"], rotation=18, ha="right")
    axes[0].set_ylabel("Failure edges")
    axes[0].set_title("Observed failures by modality and axis")
    axes[0].legend(frameon=False, fontsize=8)
    axes[0].grid(axis="y", alpha=0.2)

    global_gate_rows = fit_eligibility[fit_eligibility["scope"].eq("GLOBAL")].copy()
    colors = ["#2a9d8f" if value else "#e76f51" for value in global_gate_rows["passed"]]
    axes[1].barh(global_gate_rows["gate"], global_gate_rows["observed"].astype(float), color=colors)
    for index, row in enumerate(global_gate_rows.itertuples(index=False)):
        axes[1].text(float(row.observed), index, f"  need {row.required}", va="center", fontsize=8)
    axes[1].set_title("Global coefficient-fit gates")
    axes[1].set_xlabel("Observed count")
    axes[1].grid(axis="x", alpha=0.2)
    fig.suptitle("Stage 9 evidence structure and fit readiness", fontsize=13)
    fig.tight_layout()
    fig.savefig(FIGURE_PATH, dpi=180, bbox_inches="tight")
    plt.close(fig)

failed_gate_text = fit_eligibility.loc[~fit_eligibility["passed"], ["scope", "gate", "observed", "required"]].to_string(index=False)
summary_text = modality_summary.to_string(index=False)
decision = "FREEZE_HIERARCHICAL_DDO2_SPECIFICATION_DEFER_COEFFICIENT_FIT_AND_ACQUIRE_SEPARATE_DEVELOPMENT_EXTENSION_PLUS_LOCKED_BLIND_DATA"

report = f"""# Stage 9 — Hierarchical DDO2 Specification Freeze Report v0.1

## Decision

`{decision}`

Stage 9 verified the immutable Stage 7/8/8B lineage and audited **{len(edges)} eligible directed edges**, **{edges['modality'].nunique()} modalities**, and **{len(all_datasets)} unique datasets**. It froze three separate outputs, a partially pooled modality-aware model family, leave-one-dataset-out and leave-one-modality-out validation, and an explicit ABSTAIN contract. It did **not** estimate coefficients and did **not** perform an external blind validation.

## Observed evidence structure

```
{summary_text}
```

The edge rows are dependent because each dataset occurs in multiple source/target edges. Random edge-row splitting and naive edge-row bootstrap are prohibited.

## Why fitting is deferred

```
{failed_gate_text}
```

The frozen specification therefore remains an unfitted method contract. This prevents 21 dependent discovery edges from being presented as a deployable or externally validated DDO2.

## Frozen method

- Independent discrimination, calibration, and operating-point failure probabilities; no single global shift score.
- Shared label-free components with partially pooled modality slope deviations and directed source/target dataset random effects.
- Dataset-incident leave-one-dataset-out validation and complete leave-one-modality-out validation.
- ABSTAIN on source-gate failure, task mismatch, missing input, feature OOD, or excessive posterior uncertainty.
- Unknown modalities use the population effect with full hierarchical uncertainty; they are never silently relabelled as a known modality.

## Next boundary

New datasets must be assigned before label access to either `DEVELOPMENT_EXTENSION` or `LOCKED_BLIND_TEST`. Development data may only be added until every frozen fit gate passes. A later notebook must freeze transforms, coefficients, posterior draws, and model hash before locked-blind labels are retrieved once. At least two complete datasets or one complete modality must remain outside all feature, preprocessing, prior, coefficient, and threshold decisions.

## Integrity identifiers

- Parent Stage 8B final record: `{EXPECTED_STAGE8B_FINAL_HASH}`
- Stage 9 protocol seal: `{seal_claim}`
- Frozen model specification: `{MODEL_SPEC_HASH}`
- Frozen ABSTAIN policy: `{ABSTAIN_POLICY_HASH}`
- Future blind protocol: `{FUTURE_BLIND_PROTOCOL_HASH}`
"""
write_immutable_text(REPORT_PATH, report)

output_files = [
    PROTOCOL_SEAL_PATH,
    INPUT_COMMITMENT_PATH,
    AUDIT_ROOT / "Stage9_Edge_Library_Data_Quality_Audit_v0.1.csv",
    AUDIT_ROOT / "Stage9_Modality_Axis_Outcome_Summary_v0.1.csv",
    AUDIT_ROOT / "Stage9_Dataset_Edge_Incidence_Audit_v0.1.csv",
    AUDIT_ROOT / "Stage9_LabelFree_Component_Quality_Audit_v0.1.csv",
    AUDIT_ROOT / "Stage9_Joint_Failure_State_Audit_v0.1.csv",
    AUDIT_ROOT / "Stage9_Component_Failure_Heterogeneity_Audit_v0.1.csv",
    AUDIT_ROOT / "Stage9_Failure_Axis_Dependence_Audit_v0.1.csv",
    SPEC_ROOT / "Stage9_Frozen_DDO2_LabelFree_Input_Schema_v0.1.csv",
    SPEC_ROOT / "Stage9_Frozen_DDO2_Outcome_Axis_Definitions_v0.1.csv",
    SPEC_ROOT / "Stage9_Frozen_Hierarchical_DDO2_Model_Specification_v0.1.json",
    SPEC_ROOT / "Stage9_Frozen_DDO2_ABSTAIN_Policy_v0.1.json",
    SPEC_ROOT / "Stage9_Frozen_Excluded_Component_Register_v0.1.csv",
    SPEC_ROOT / "Stage9_Frozen_Unfitted_DDO2_State_v0.1.json",
    VALIDATION_ROOT / "Stage9_Frozen_Grouped_Validation_Fold_Registry_v0.1.csv",
    VALIDATION_ROOT / "Stage9_DDO2_Coefficient_Fit_Eligibility_Audit_v0.1.csv",
    BLIND_ROOT / "Stage9_Axis_Class_Balance_Shortfall_v0.1.csv",
    BLIND_ROOT / "Stage9_Stage10_Acquisition_Role_Registry_Template_v0.1.csv",
    BLIND_ROOT / "Stage9_Future_Blind_Edge_LabelFree_Input_Template_v0.1.csv",
    BLIND_ROOT / "Stage9_Future_Blind_DDO2_Prediction_Freeze_Template_v0.1.csv",
    BLIND_ROOT / "Stage9_Frozen_Future_Blind_Validation_Protocol_v0.1.json",
    BLIND_ROOT / "Stage9_Next_Stage_Data_Requirement_Summary_v0.1.csv",
    REPORT_PATH,
    FIGURE_PATH,
]
assert all(path.is_file() for path in output_files)
output_integrity = pd.DataFrame([
    {
        "relative_path": str(path.relative_to(STAGE9_ROOT)),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in sorted(output_files, key=lambda item: str(item))
])
write_immutable_csv(OUTPUT_MANIFEST_PATH, output_integrity)

tracked_stage9_bytes = sum(path.stat().st_size for path in output_files + [OUTPUT_MANIFEST_PATH])
assert tracked_stage9_bytes <= MAXIMUM_NEW_STAGE9_BYTES

if FINAL_RECORD_PATH.is_file():
    final_record, final_claim = verify_self_hashed_json(FINAL_RECORD_PATH)
    assert final_record["decision"] == decision
    assert final_record["stage9_protocol_seal_sha256"] == seal_claim
    assert final_record["model_specification_sha256"] == MODEL_SPEC_HASH
    assert final_record["fit_authorised"] is False
else:
    final_record = {
        "stage": "Stage9",
        "decision": decision,
        "scope": "SPECIFICATION_FREEZE_NOT_COEFFICIENT_FIT_NOT_EXTERNAL_VALIDATION",
        "parent_stage7_final_record_sha256": EXPECTED_STAGE7_FINAL_HASH,
        "parent_stage8_final_record_sha256": EXPECTED_STAGE8_FINAL_HASH,
        "parent_stage8b_final_record_sha256": EXPECTED_STAGE8B_FINAL_HASH,
        "stage9_protocol_seal_sha256": seal_claim,
        "notebook_source_sha256": notebook_source_hash,
        "edge_library_sha256": sha256_file(EDGE_LIBRARY_PATH),
        "eligible_edges": int(len(edges)),
        "modalities": int(edges["modality"].nunique()),
        "unique_datasets": int(len(all_datasets)),
        "model_specification_sha256": MODEL_SPEC_HASH,
        "abstain_policy_sha256": ABSTAIN_POLICY_HASH,
        "unfitted_model_state_sha256": UNFITTED_STATE_HASH,
        "future_blind_protocol_sha256": FUTURE_BLIND_PROTOCOL_HASH,
        "fit_authorised": False,
        "final_ddo2_fitted": False,
        "future_data_accessed": False,
        "future_blind_labels_accessed": False,
        "final_external_validation_performed": False,
        "target_model_refit": False,
        "threshold_tuned": False,
        "single_global_score_fitted": False,
        "tracked_stage9_output_bytes_excluding_runtime_and_final_record": int(tracked_stage9_bytes),
        "maximum_new_stage9_bytes": int(MAXIMUM_NEW_STAGE9_BYTES),
        "output_integrity_manifest_sha256": sha256_file(OUTPUT_MANIFEST_PATH),
        "next_step": "ACQUIRE_PREASSIGNED_DEVELOPMENT_EXTENSION_AND_LOCKED_BLIND_DATA_UNDER_THE_FROZEN_ROLE_REGISTRY",
        "completed_utc": utc_now(),
    }
    final_record["final_record_sha256"] = sha256_json(final_record)
    final_claim = final_record["final_record_sha256"]
    write_immutable_json(FINAL_RECORD_PATH, final_record)

runtime_state.update({
    "completed": True,
    "decision": decision,
    "fit_authorised": False,
    "final_ddo2_fitted": False,
    "final_blind_validation_performed": False,
    "final_record_sha256": final_claim,
    "last_updated_utc": utc_now(),
})
atomic_json(RUNTIME_STATE_PATH, runtime_state)

print("\n================ STAGE 9 HIERARCHICAL DDO2 SPECIFICATION FREEZE COMPLETE ================")
print("Eligible edges / modalities / datasets:", len(edges), "/", edges["modality"].nunique(), "/", len(all_datasets))
print("Coefficient fit authorised:", False)
print("Final DDO2 fitted / blind validation performed:", False, "/", False)
print("Decision:", decision)
print("Protocol seal:", seal_claim)
print("Model specification hash:", MODEL_SPEC_HASH)
print("Final record hash:", final_claim)
actual_stage9_bytes = sum(path.stat().st_size for path in STAGE9_ROOT.rglob("*") if path.is_file())
print("Actual Stage 9 bytes:", actual_stage9_bytes)
print("Next step:", final_record["next_step"])



================ STAGE 9 HIERARCHICAL DDO2 SPECIFICATION FREEZE COMPLETE ================
Eligible edges / modalities / datasets: 21 / 3 / 10
Coefficient fit authorised: False
Final DDO2 fitted / blind validation performed: False / False
Decision: FREEZE_HIERARCHICAL_DDO2_SPECIFICATION_DEFER_COEFFICIENT_FIT_AND_ACQUIRE_SEPARATE_DEVELOPMENT_EXTENSION_PLUS_LOCKED_BLIND_DATA
Protocol seal: f5689995ec6ab0dcf0bcacf813d578edd47edbedf8fb96b7809ec674e3cad593
Model specification hash: e2a0d8f65143e0b9d58d740595a39b803f1684481c4f8af01f05935bfaacb732
Final record hash: 2568a9fdaff83ab74938655d65d25cd9d50f8b28bafbc44e856098932f6c6648
Actual Stage 9 bytes: 167648
Next step: ACQUIRE_PREASSIGNED_DEVELOPMENT_EXTENSION_AND_LOCKED_BLIND_DATA_UNDER_THE_FROZEN_ROLE_REGISTRY
